In [ ]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('data/customers-100000.csv')
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   Index              100000 non-null  int64
 1   Customer Id        100000 non-null  str  
 2   First Name         100000 non-null  str  
 3   Last Name          100000 non-null  str  
 4   Company            100000 non-null  str  
 5   City               100000 non-null  str  
 6   Country            100000 non-null  str  
 7   Phone 1            100000 non-null  str  
 8   Phone 2            100000 non-null  str  
 9   Email              100000 non-null  str  
 10  Subscription Date  100000 non-null  str  
 11  Website            100000 non-null  str  
dtypes: int64(1), str(11)
memory usage: 9.2 MB
None


In [6]:
# Data cleaning
df.dropna(inplace=True)
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   Index              100000 non-null  int64
 1   Customer Id        100000 non-null  str  
 2   First Name         100000 non-null  str  
 3   Last Name          100000 non-null  str  
 4   Company            100000 non-null  str  
 5   City               100000 non-null  str  
 6   Country            100000 non-null  str  
 7   Phone 1            100000 non-null  str  
 8   Phone 2            100000 non-null  str  
 9   Email              100000 non-null  str  
 10  Subscription Date  100000 non-null  str  
 11  Website            100000 non-null  str  
dtypes: int64(1), str(11)
memory usage: 9.2 MB
None


### 1. Subscription per month and growth

In [ ]:
df['Subscription Date'] = pd.to_datetime(df['Subscription Date'])

monthly_signups = df.set_index('Subscription Date').resample('ME').size()

mom_growth = monthly_signups.pct_change().fillna(0) * 100

velocity_df = pd.DataFrame({
    'New Subscriptions': monthly_signups,
    'MoM Growth (%)': mom_growth.round(2)
})
print(velocity_df.tail(5))

                   New Subscriptions  MoM Growth (%)
Subscription Date                                   
2022-01-31                      3523            0.06
2022-02-28                      3178           -9.79
2022-03-31                      3564           12.15
2022-04-30                      3392           -4.83
2022-05-31                      3234           -4.66


### 2. Top 5 countries by total subscribers

In [18]:
top_5_countries = df['Country'].value_counts().sort_values(ascending=False).head(5).astype(int)

# top_markets_df = df[df['Country'].isin(top_5_countries)]

# city_density = top_markets_df.groupby(['Country', 'City']).size().reset_index(name='User Count')
# top_cities = city_density.sort_values(['Country', 'User Count'], ascending=[True, False]) \
#                           .groupby('Country').head(2)

print(top_5_countries)

Country
Congo               835
Korea               820
Saudi Arabia        463
Pitcairn Islands    456
Saint Martin        453
Name: count, dtype: int64


### 3. B2B or B2C?

In [ ]:
free_domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'icloud.com', 'aol.com']

df['Email_Domain'] = df['Email'].str.split('@').str[1].str.lower()

df['Customer_Type'] = np.where(df['Email_Domain'].isin(free_domains), 'B2C (Consumer)', 'B2B (Corporate)')

archetype_mix = df['Customer_Type'].value_counts(normalize=True) * 100

print(archetype_mix.round(2).astype(str) + '%')

Customer_Type
B2B (Corporate)    100.0%
Name: proportion, dtype: str
